In [1]:
from pathlib import Path
import sys
import pandas as pd 
import logging
import os 
logger = logging.getLogger(__name__)
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
%load_ext autoreload 
%autoreload 2 

In [3]:
import yaml

from src.data import (
    create_db_engine,
    test_db_connection,
    get_table_names,
    load_all_tables,
    build_ml_table,
    save_dataframe,
)

In [4]:
with open(
    PROJECT_ROOT / "config" / "config.yaml",
    "r",
    encoding="utf-8",
) as file:
    config = yaml.safe_load(file)

In [5]:
import sys 
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

In [6]:
engine = create_db_engine(config)

if test_db_connection(engine):
    logger.info("Database connection successful.")
else:
    logger.error("Database connection failed.",exc_info=True)

2026-09-21 14:56:25,627 - INFO - Database connection successful.


In [7]:
required_tables = [
    "orders",
    "customers",
    "products",
    "sellers",
    "order_items",
    "order_payments",
    "order_reviews",
    "product_category_translation",
    "geolocation",
]

tables = load_all_tables(
    engine,
    required_tables,
)

In [8]:
orders = tables["orders"]
customers = tables["customers"]
products = tables["products"]
sellers = tables["sellers"]
order_items = tables["order_items"]
order_payments = tables["order_payments"]
order_reviews = tables["order_reviews"]
category_translation = tables[
    "product_category_translation"
]
geolocation = tables["geolocation"]

In [9]:
ml_table = build_ml_table(
    orders=orders,
    customers=customers,
    order_items=order_items,
    order_payments=order_payments,
)

In [10]:
logger.info(f"Rows: %s", {len(ml_table)})
logger.info(
    "Unique orders: %s",
    ml_table["order_id"].nunique(),
)
logger.info(
    "Duplicate order IDs: %s",
    ml_table["order_id"].duplicated().sum(),
)

2026-09-21 14:59:32,434 - INFO - Rows: {99441}
2026-09-21 14:59:32,615 - INFO - Unique orders: 99441
2026-09-21 14:59:32,792 - INFO - Duplicate order IDs: 0


In [11]:
from src.validation import (
    get_table_shapes,
    validate_primary_keys,
    validate_composite_keys,
    validate_ml_table,
    get_missing_values,
    validate_no_duplicate_orders,
)

In [12]:
table_shapes = get_table_shapes(tables)

table_shapes

,table,rows,columns
0,orders,99441,8
1,customers,99441,5
2,products,32951,9
3,sellers,3095,4
4,order_items,112650,7
5,order_payments,103886,5
6,order_reviews,70000,7
7,product_category_translation,71,2
8,geolocation,1000163,5


In [13]:
primary_key_results = validate_primary_keys(
    orders=orders,
    customers=customers,
    products=products,
    sellers=sellers,
)

primary_key_results

{'orders.order_id': 0,
 'customers.customer_id': 0,
 'products.product_id': 0,
 'sellers.seller_id': 0}

In [14]:
composite_key_results = validate_composite_keys(
    order_items=order_items,
    order_payments=order_payments,
)

composite_key_results

{'order_items.(order_id, order_item_id)': 0,
 'order_payments.(order_id, payment_sequential)': 0}

In [15]:
ml_table = build_ml_table(
    orders=orders,
    customers=customers,
    order_items=order_items,
    order_payments=order_payments,
)

In [16]:
ml_validation = validate_ml_table(ml_table)

ml_validation

{'rows': 99441,
 'columns': 22,
 'unique_orders': 99441,
 'duplicate_order_ids': 0}

In [17]:
from src.features import create_features

In [18]:
ml_table_features = create_features(
    ml_table=ml_table,
    order_items=order_items,
    products=products,
    sellers=sellers,
    geolocation=geolocation,
)

In [19]:
logger.info("Rows: %s", len(ml_table_features))
logger.info(
    "Unique orders: %store",
    ml_table_features["order_id"].nunique()
)
logger.info(
    "Duplicate order IDs: %s",
    ml_table_features["order_id"].duplicated().sum()
)

2026-09-21 15:02:00,739 - INFO - Rows: 99441
2026-09-21 15:02:00,913 - INFO - Unique orders: 99441tore
2026-09-21 15:02:00,970 - INFO - Duplicate order IDs: 0


In [20]:
new_features = [
    "purchase_year",
    "purchase_month",
    "purchase_day",
    "purchase_weekday",
    "purchase_hour",
    "is_weekend",
    "multi_seller",
    "freight_ratio",
    "freight_per_item",
    "total_weight",
    "total_volume",
    "avg_volume",
    "max_volume",
    "distance_km",
    "is_cross_state",
]

ml_table_features[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
purchase_year,99441.0,2017.539838,0.505007,2016.0,2017.000000,2018.000000,2018.000000,2.018000e+03
purchase_month,99441.0,6.032220,3.232999,1.0,3.000000,6.000000,8.000000,1.200000e+01
purchase_day,99441.0,15.505948,8.667298,1.0,8.000000,15.000000,23.000000,3.100000e+01
purchase_weekday,99441.0,2.755735,1.966495,0.0,1.000000,3.000000,4.000000,6.000000e+00
purchase_hour,99441.0,14.770879,5.326666,0.0,11.000000,15.000000,19.000000,2.300000e+01
is_weekend,99441.0,0.229754,0.420677,0.0,0.000000,0.000000,0.000000,1.000000e+00
multi_seller,99441.0,0.012852,0.112636,0.0,0.000000,0.000000,0.000000,1.000000e+00
freight_ratio,98666.0,0.308389,0.314762,0.0,0.131864,0.224374,0.380191,2.144706e+01
freight_per_item,98666.0,20.190469,15.797846,0.0,13.370000,16.360000,21.180000,4.096800e+02
total_weight,98666.0,2390.027669,4773.239825,0.0,300.000000,750.000000,2066.750000,1.844000e+05


In [21]:
ml_table_features[new_features].isna().sum().sort_values(
    ascending=False
)

distance_km         1264
avg_volume           791
max_volume           791
freight_ratio        775
freight_per_item     775
total_weight         775
total_volume         775
purchase_year          0
purchase_month         0
purchase_day           0
purchase_weekday       0
purchase_hour          0
is_weekend             0
multi_seller           0
is_cross_state         0
dtype: int64

In [22]:
output_path = PROJECT_ROOT / config["paths"]["ml_table"]

save_dataframe(
    ml_table_features,
    output_path,
)

logger.info(f"Saved to: {output_path}")

2026-09-21 15:02:36,132 - INFO - Saved to: c:\Users\Anwar Altorkmani\Desktop\MLOps_Project_New\data\raw\olist_table.csv


In [23]:
import pandas as pd 
final_check = pd.read_csv(output_path)

logger.info("Shape: %s", final_check.shape)
logger.info(
    "Unique orders: %s",
    final_check["order_id"].nunique()
)
logger.info(
    "Duplicate order IDs: %s",
    final_check["order_id"].duplicated().sum()
)

2026-09-21 15:03:09,445 - INFO - Shape: (99441, 37)
2026-09-21 15:03:09,583 - INFO - Unique orders: 99441
2026-09-21 15:03:09,697 - INFO - Duplicate order IDs: 0
